# 02 — Train the base detector (CPU)

Run these cells in order after installing the replacement `scripts/train_detector.py`. This notebook runs a **smoke test only**: 200 training images for one epoch. It uses a fresh Python process and does not train CLIP or use the test split. See the final cell for the separate full-training command.


In [1]:
from pathlib import Path
import os
import sys

current = Path.cwd().resolve()
repo_root = next((p for p in [current, *current.parents]
                  if (p / "configs/config.yaml").is_file()
                  and (p / "scripts/train_detector.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from inside your openworld project folder.")
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
print("Project folder:", repo_root)
print("Notebook Python:", sys.executable)


Project folder: C:\Users\karki\Downloads\main_openworld\openworld
Notebook Python: c:\Users\karki\Downloads\main_openworld\openworld\.venv\Scripts\python.exe


In [2]:
# Verify the stripped annotations and files before starting training.
from src.openworld.categories import label_map_for_base
import src.openworld.categories as categories

for item in ["configs/config.yaml", "scripts/train_detector.py",
             "data/annotations/instances_train2017.stripped.json",
             "data/splits/train_image_ids.json", "data/images/train2017"]:
    if not Path(item).exists():
        raise FileNotFoundError(f"Missing required path: {(repo_root / item).resolve()}")
label_map, base_names = label_map_for_base(
    "data/annotations/instances_train2017.stripped.json"
)
if len(base_names) != 15 or sorted(label_map.values()) != list(range(1, 16)):
    raise ValueError("Expected 15 base-class labels numbered 1 through 15.")
print("Category helper:", categories.__file__)
print("Base classes:", len(base_names), "(plus background = 16 detector outputs)")
print("Checks passed.")


Category helper: C:\Users\karki\Downloads\main_openworld\openworld\src\openworld\categories.py
Base classes: 15 (plus background = 16 detector outputs)
Checks passed.


In [3]:
# Smoke test in a fresh Python process. The first run may download the
# ImageNet ResNet-50 backbone weights, not the COCO dataset.
import subprocess
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
smoke_dir = Path("outputs/checkpoints/smoke") / run_id
smoke_log = Path("outputs/metrics") / f"smoke_{run_id}.csv"
command = [sys.executable, "-u", "scripts/train_detector.py",
           "--mode", "train", "--config", "configs/config.yaml",
           "--images-root", "data/images/train2017",
           "--limit-images", "200", "--epochs", "1", "--batch-size", "2",
           "--lr", "0.005", "--checkpoint-dir", str(smoke_dir),
           "--log-file", str(smoke_log)]
print("Smoke checkpoint folder:", smoke_dir, flush=True)
print("Starting training; the first batch can take some time on CPU.\n", flush=True)
process = subprocess.Popen(command, cwd=repo_root, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
    status = process.wait()
except BaseException:
    process.terminate()
    process.wait()
    raise
if status != 0:
    raise RuntimeError(f"Training exited with code {status}; read the output above.")
print("\nSmoke training succeeded.")


Smoke checkpoint folder: outputs\checkpoints\smoke\20261112_194011_240305
Starting training; the first batch can take some time on CPU.

loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
C:\Users\karki\Downloads\main_openworld\openworld\src\openworld\engine.py:48: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  loss_sum[k] = loss_sum.get(k, 0.0) + float(v)
[epoch 1] iter 50 | loss_classifier=1.1924 loss_box_reg=0.1047 loss_objectness=0.5523 loss_rpn_box_reg=0.1572 | 453.2s
[epoch 1] iter 100 | loss_classifier=0.7380 loss_box_reg=0.1411 loss_objectness=0.4061 loss_rpn_box_reg=0.1385 | 905.8s
Saved checkpoint epoch 1: loss_classifier=0.7380 loss_box_reg=0.1411 loss_objectness=0.4061 loss_rpn_box_reg=0.1385
Training complete. Checkpoints

In [6]:
# Run after the smoke test finishes successfully.
import csv
checkpoint = smoke_dir / "epoch_1.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint}")
with smoke_log.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
if not rows:
    raise RuntimeError("The training log is empty.")
print("Checkpoint:", checkpoint.resolve())
print("Log:", smoke_log.resolve())
print("Epoch results:")
for row in rows:
    print(row)
print("\nThe smoke test logs one epoch; it cannot establish a decreasing loss trend.")


Checkpoint: C:\Users\karki\Downloads\main_openworld\openworld\outputs\checkpoints\smoke\20261112_194011_240305\epoch_1.pt
Log: C:\Users\karki\Downloads\main_openworld\openworld\outputs\metrics\smoke_20261112_194011_240305.csv
Epoch results:
{'epoch': '1', 'iteration': '100', 'loss_classifier': '0.73803', 'loss_box_loss': '0.0', 'loss_objectness': '0.40611', 'loss_rpn_box_loss': '0.0', 'lr': '0.0025'}

The smoke test logs one epoch; it cannot establish a decreasing loss trend.


## Next: full training

Only after the smoke test succeeds, open a PowerShell terminal in the **project root** and run:

```powershell
.\.venv\Scripts\python.exe -u scripts\train_detector.py --mode train --config configs\config.yaml --images-root data\images\train2017 --epochs 3 --batch-size 2 --lr 0.005 --checkpoint-dir outputs\checkpoints --log-file outputs\metrics\full_train_log.csv
```

Full training may take hours on CPU. It saves `epoch_1.pt` through `epoch_3.pt` under `outputs/checkpoints` and may overwrite checkpoints with those names. Use a different checkpoint folder if needed. The training script does not automatically run evaluation. Do not tune on `val_test`.
